# 02. QA- Data Quality & Descriptive Analysis

## Objetivo
Realizar un análisis descriptivo y controles de calidad sobre los datasets de plataformas de streaming.
Se documentan hallazgos, riesgos y propuestas de solución.

## Datos analizados
- Archivos: `data/*_titles.csv`
- Carga automática por archivo (plataforma inferida del nombre)
- Delimitador detectado automáticamente por archivo

# Librerias

In [11]:
import pandas as pd
import csv
from pathlib import Path
import numpy as np
import re

# Funciones

In [12]:
def sniff_sep(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="replace")[:5000]
    dialect = csv.Sniffer().sniff(sample, delimiters=[",",";","\t","|"])
    return dialect.delimiter

def platform_from_filename(path: Path) -> str:
    name = path.stem  
    suffix = "_titles"
    if not name.endswith(suffix):
        raise ValueError(f"Nombre inválido: {path.name}. Se espera formato <plataforma>_titles.csv")
    return name[:-len(suffix)]  # todo antes de "_titles"

def parse_duration(s):
    if pd.isna(s) or str(s).strip()=="":
        return (np.nan, np.nan)
    s = str(s).strip()

    m = re.match(r"^(\d+)\s*min$", s, flags=re.IGNORECASE)
    if m:
        return (int(m.group(1)), np.nan)

    m = re.match(r"^(\d+)\s*Season[s]?$", s, flags=re.IGNORECASE)
    if m:
        return (np.nan, int(m.group(1)))

    return (np.nan, np.nan)

def sniff_sep(path: Path) -> str:
    sample = path.read_text(encoding="utf-8", errors="replace")[:5000]
    dialect = csv.Sniffer().sniff(sample, delimiters=[",",";","\t","|"])
    return dialect.delimiter

def platform_from_filename(path: Path) -> str:
    name = path.stem  
    suffix = "_titles"
    if not name.endswith(suffix):
        raise ValueError(f"Nombre inválido: {path.name}. Se espera formato <plataforma>_titles.csv")
    return name[:-len(suffix)]  # todo antes de "_titles"


# Lectura de archivos
Iteramos los archivos encontrads y no hardcodeados para dejar la solucion escalable a que se agreguen otras plataformas.

Detectamos los separadores automaticamente para dejar la solucion mas robusta.

In [3]:
import pandas as pd
import csv
from pathlib import Path



DATA_DIR = Path("../data")  
files = sorted(DATA_DIR.glob("*_titles.csv"))


#Validamos que existan archivos
if not files:
    raise FileNotFoundError(f"No se encontraron archivos *_titles.csv en {DATA_DIR.resolve()}")

dfs = []
seps_found = {}

for f in files:
    sep = sniff_sep(f)
    seps_found[f.name] = sep

    tmp = pd.read_csv(f, sep=sep)
    #Agregamos el nombre del archivo de para trazabilidad de los datos
    tmp["platform"] = platform_from_filename(f)
    tmp["source_file"] = f.name
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

print("Files loaded:", [f.name for f in files])
print("Separators detected:", {k: repr(v) for k, v in seps_found.items()})
print("Platforms:", sorted(df["platform"].unique().tolist()))
print("df shape:", df.shape)

df.head()

Files loaded: ['disney_plus_titles.csv', 'netflix_titles.csv']
Separators detected: {'disney_plus_titles.csv': "','", 'netflix_titles.csv': "';'"}
Platforms: ['disney_plus', 'netflix']
df shape: (10259, 14)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,platform,source_file
0,s1,Movie,Duck the Halls: A Mickey Mouse Christmas Special,"Alonso Ramirez Ramos, Dave Wasson","Chris Diamantopoulos, Tony Anselmo, Tress MacN...",NaN,"November 26, 2021",2016,TV-G,23 min,"Animation, Family",Join Mickey and the gang as they duck the halls!,disney_plus,disney_plus_titles.csv
1,s2,Movie,Ernest Saves Christmas,John Cherry,"Jim Varney, Noelle Parker, Douglas Seale",NaN,"November 26, 2021",1988,PG,91 min,Comedy,Santa Claus passes his magic bag to a new St. ...,disney_plus,disney_plus_titles.csv
2,s3,Movie,Ice Age: A Mammoth Christmas,Karen Disher,"Raymond Albert Romano, John Leguizamo, Denis L...",United States,"November 26, 2021",2011,TV-G,23 min,"Animation, Comedy, Family",Sid the Sloth is on Santa's naughty list.,disney_plus,disney_plus_titles.csv
3,s4,Movie,The Queen Family Singalong,Hamish Hamilton,"Darren Criss, Adam Lambert, Derek Hough, Alexa...",NaN,"November 26, 2021",2021,TV-PG,41 min,Musical,"This is real life, not just fantasy!",disney_plus,disney_plus_titles.csv
4,s5,TV Show,The Beatles: Get Back,NaN,"John Lennon, Paul McCartney, George Harrison, ...",NaN,"November 25, 2021",2021,NaN,1 Season,"Docuseries, Historical, Music",A three-part documentary from Peter Jackson ca...,disney_plus,disney_plus_titles.csv


# Verificamos columnas y tipos

In [4]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.dtypes

shape: (10259, 14)
columns: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description', 'platform', 'source_file']


show_id            str
type               str
title              str
director           str
cast               str
country            str
date_added         str
release_year    object
rating             str
duration           str
listed_in          str
description        str
platform           str
source_file        str
dtype: object

## Hallazgos - Valores nulos por columna (por plataforma)

La tabla muestra la **tasa de valores nulos** (0 a 1) para cada columna, separada por plataforma (`disney_plus` vs `netflix`).

### Principales hallazgos
- **Campos con mayor incompletitud (metadata):**
  - `director`: ~32.6% nulos en **disney_plus** y ~29.9% en **netflix** (es el campo más incompleto en ambas).
  - `country`: ~15.1% nulos en **disney_plus** vs ~9.5% en **netflix**.
  - `cast`: ~13.1% nulos en **disney_plus** vs ~9.4% en **netflix**.

- **Campos casi completos (baja tasa de nulos):**
  - `date_added` y `rating`: ~0.2% o menos en **disney_plus** y ~0.14% o menos en **netflix**.
  - `duration`, `listed_in`, `description`, `title`, `release_year`, `type`: prácticamente completos (cercanos a 0%; con nulos mínimos en netflix).

- **Campos completos (0% nulos):**
  - `show_id` y `source_file` no presentan nulos en ninguna plataforma (útiles para clave y trazabilidad).


In [7]:
null_by_platform = df.groupby("platform").apply(lambda x: x.isna().mean()).T
null_by_platform.sort_values(by=list(null_by_platform.columns), ascending=False).head(15)

platform,disney_plus,netflix
director,0.326207,0.299239
country,0.151034,0.094562
cast,0.131034,0.093768
date_added,0.002069,0.001362
rating,0.002069,0.000681
duration,0.000000,0.000568
listed_in,0.000000,0.000341
description,0.000000,0.000341
title,0.000000,0.000227
release_year,0.000000,0.000227


## Duplicados y clave natural

No se encontraron duplicados por id. Se valida `platform + show_id` como clave primaria candidata para el modelo.



In [9]:
# Duplicados exactos por clave natural dentro de cada plataforma
dup_count = df.duplicated(subset=["platform", "show_id"]).sum()
print("Duplicados platform+show_id:", dup_count)

if dup_count > 0:
    dups = df[df.duplicated(subset=["platform","show_id"], keep=False)].sort_values(["platform","show_id"])
    display(dups.head(20))

Duplicados platform+show_id: 0


In [10]:
soft = df.duplicated(subset=["platform","title","type","release_year"], keep=False)
soft_dups = df[soft].sort_values(["platform","title","release_year"])
print("Soft duplicates (platform+title+type+release_year):", len(soft_dups))
display(soft_dups.head(20))

Soft duplicates (platform+title+type+release_year): 0


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,platform,source_file


## Calidad de `duration` (minutos vs temporadas)

**Contexto:** `duration` mezcla unidades:
- Para **Movie** suele venir en minutos (`"90 min"`).
- Para **TV Show** suele venir en temporadas (`"2 Seasons"`).

**Check realizado:**
1. Parseo a dos columnas: `duration_minutes` y `seasons`.
2. Validación de consistencia:
   - Movie → debe poblar `duration_minutes`
   - TV Show → debe poblar `seasons`

**Riesgo si no se trata:** consultas como “top películas más largas” pueden dar resultados incorrectos o excluir registros.

**Solución propuesta:**
- Persistir `duration_raw` + campos derivados (`duration_minutes`, `seasons`).
- Aplicar reglas por tipo y reportar los valores no parseables.

In [13]:
parsed = df["duration"].apply(parse_duration)
df["duration_minutes"] = parsed.apply(lambda t: t[0])
df["seasons"] = parsed.apply(lambda t: t[1])

# Checks de consistencia por tipo
movies_bad = (df["type"] == "Movie") & df["duration"].notna() & df["duration_minutes"].isna()
tv_bad     = (df["type"] == "TV Show") & df["duration"].notna() & df["seasons"].isna()

print("Movies con duration no parseable:", int(movies_bad.sum()))
print("TV Shows con duration no parseable:", int(tv_bad.sum()))

# Ejemplos problematicos (si existen)
if (movies_bad | tv_bad).any():
    display(df.loc[movies_bad | tv_bad, ["platform","show_id","title","type","duration","source_file"]].head(20))

# Valores no parseables (distintos) para reportar
bad_values = df.loc[movies_bad | tv_bad, "duration"].dropna().astype(str).str.strip().value_counts().head(10)
display(bad_values)

Movies con duration no parseable: 0
TV Shows con duration no parseable: 0


Series([], Name: count, dtype: int64)

## Calidad de `date_added`

**Check realizado:**
- Se preserva `date_added_raw` (valor original).
- Se intenta parsear a fecha en `date_added_parsed` usando el formato esperado (`"%B %d, %Y"`).
- Se cuantifican fallos de parseo y se listan ejemplos para diagnóstico.

**Riesgo:** si `date_added` no se parsea correctamente, análisis temporales (ingesta, tendencias, deduplicación por “más reciente”) pueden ser inconsistentes.

**Solución propuesta:**
- Persistir `date_added_raw` + `date_added_parsed`.
- Loguear valores no parseables y, si son sistemáticos, ampliar la lógica de parseo (formatos alternativos).

In [15]:
# Guardamos raw para trazabilidad
df["date_added_raw"] = df["date_added"]

# Parse robusto (si viene como "September 9, 2019")
df["date_added_parsed"] = pd.to_datetime(
    df["date_added_raw"].astype(str).str.strip(),
    format="%B %d, %Y",
    errors="coerce"
)

# Métricas de parseo
has_raw = df["date_added_raw"].notna() & (df["date_added_raw"].astype(str).str.strip() != "")
bad_parse = has_raw & df["date_added_parsed"].isna()

print("date_added no nulo (raw):", int(has_raw.sum()))
print("date_added parse fallido:", int(bad_parse.sum()))
print("parse success rate:", round(1 - bad_parse.sum()/max(has_raw.sum(),1), 4))

# Ejemplos de valores que no parsean (si existen)
if bad_parse.any():
    display(df.loc[bad_parse, ["platform","show_id","title","date_added_raw","source_file"]].head(20))
    display(df.loc[bad_parse, "date_added_raw"].astype(str).str.strip().value_counts().head(10))

# Rango y distribución (sanity check)
display(df["date_added_parsed"].describe())

date_added no nulo (raw): 10244
date_added parse fallido: 1
parse success rate: 0.9999


,platform,show_id,title,date_added_raw,source_file
9871,netflix,"Flying Fortress""",NaN,TV-PG,netflix_titles.csv


date_added_raw
TV-PG    1
Name: count, dtype: int64

count                         10243
mean     2019-07-08 17:04:26.054866
min             2008-01-01 00:00:00
25%             2018-07-26 00:00:00
50%             2019-11-12 00:00:00
75%             2020-09-01 00:00:00
max             2021-11-26 00:00:00
Name: date_added_parsed, dtype: object

## Hallazgos — Columnas multi-valued (`cast`, `director`, `listed_in`, `country`)

Se analizaron columnas que contienen **listas separadas por coma**, calculando la cantidad de elementos por registro.

### `cast` (actores)
- **Mediana:** 7 actores por título (P25=3, P75=10).
- **Máximo:** 50 actores en un registro.
- **Interpretación:** relación claramente **muchos-a-muchos** (título ↔ personas). Existe alta variabilidad y outliers con casts muy grandes.

### `director` (directores)
- **Mediana:** 1 director por título (P25=0, P75=1).
- **Máximo:** 13 directores.
- **Interpretación:** la mayoría de títulos tiene **0–1 director**, pero hay casos excepcionales con múltiples directores (requiere modelar muchos-a-muchos igualmente).

### `listed_in` (géneros/categorías)
- **Mediana:** 2 categorías por título (P25=2, P75=3).
- **Máximo:** 3 categorías.
- **Interpretación:** distribución bastante estable (2–3 géneros por registro). Es un buen candidato a dimensión `genre`.

### `country` (país/es)
- **Mediana:** 1 país por título (P25=1, P75=1).
- **Máximo:** 15 países.
- **Interpretación:** normalmente hay 1 país, pero existen registros multi-país (también muchos-a-muchos). Esto puede impactar análisis geográficos (un título puede contarse en múltiples países).

### Implicancias para QA / análisis
- Mantener estas columnas como strings dificulta:
  - consultas tipo “top actores”, “top directores”, “top géneros”,
  - controles de calidad (tokens vacíos, espacios, inconsistencias),
  - conteos consistentes (riesgo de sub/sobre-conteo si no se normaliza).

### Solución propuesta
- Normalizar a tablas puente y dimensiones:
  - `person` + `title_person` (roles: actor/director)
  - `genre` + `title_genre` (`listed_in`)
  - `country_dim` + `title_country` (`country`)
- Aplicar limpieza mínima previa: `strip()` y filtrado de tokens vacíos.

In [16]:
multi_cols = [c for c in ["cast", "director", "listed_in", "country"] if c in df.columns]
print("multi-valued cols:", multi_cols)

def split_count(x):
    if pd.isna(x) or str(x).strip()=="":
        return 0
    return len([p.strip() for p in str(x).split(",") if p.strip()])

for c in multi_cols:
    df[c + "_count"] = df[c].apply(split_count)
    print("\n", c)
    display(df[c + "_count"].describe())

multi-valued cols: ['cast', 'director', 'listed_in', 'country']

 cast


count    10259.000000
mean         6.825909
std          4.732801
min          0.000000
25%          3.000000
50%          7.000000
75%         10.000000
max         50.000000
Name: cast_count, dtype: float64


 director


count    10259.000000
mean         0.789453
std          0.688117
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max         13.000000
Name: director_count, dtype: float64


 listed_in


count    10259.000000
mean         2.264451
std          0.776278
min          0.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          3.000000
Name: listed_in_count, dtype: float64


 country


count    10259.000000
mean         1.124866
std          0.773731
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.000000
Name: country_count, dtype: float64

## Top valores “más largos”

In [17]:
for c in multi_cols:
    print(f"\nTop {c} by count")
    display(
        df.sort_values(c + "_count", ascending=False)[
            ["platform","show_id","title",c,c + "_count","source_file"]
        ].head(10)
    )


Top cast by count


,platform,show_id,title,cast,cast_count,source_file
3304,netflix,s1855,Social Distance,"Danielle Brooks, Oscar Nuñez, Mike Colter, Hea...",50,netflix_titles.csv
5224,netflix,s3775,Black Mirror,"Jesse Plemons, Cristin Milioti, Jimmi Simpson,...",50,netflix_titles.csv
3089,netflix,s1640,Heartbreak High,"Callan Mulvey, Lara Cox, Emma Roche, Ada Nicod...",47,netflix_titles.csv
4899,netflix,s3450,Creeped Out,"Victoria Diamond, William Romain, Sydney Wade,...",47,netflix_titles.csv
5670,netflix,s4221,COMEDIANS of the world,"Neal Brennan, Chris D'Elia, Nicole Byer, Nick ...",47,netflix_titles.csv
7636,netflix,s6187,Arthur Christmas,"James McAvoy, Hugh Laurie, Bill Nighy, Jim Bro...",44,netflix_titles.csv
6755,netflix,s5306,Narcos,"Wagner Moura, Pedro Pascal, Boyd Holbrook, Dam...",42,netflix_titles.csv
4688,netflix,s3239,Dolly Parton's Heartstrings,"Dolly Parton, Julianne Hough, Kimberly William...",41,netflix_titles.csv
7063,netflix,s5614,"Michael Bolton's Big, Sexy Valentine's Day Spe...","Michael Bolton, Andy Samberg, Will Forte, Kenn...",41,netflix_titles.csv
2333,netflix,s884,"Love, Death & Robots","Topher Grace, Mary Elizabeth Winstead, Gary Co...",40,netflix_titles.csv



Top director by count


,platform,show_id,title,director,director_count,source_file
7337,netflix,s5888,Walt Disney Animation Studios Short Films Coll...,"Chris Buck, Jennifer Lee, Patrick Osborne, Lau...",13,netflix_titles.csv
8965,netflix,s7516,Movie 43,"Peter Farrelly, Will Graham, Steve Carr, Griff...",12,netflix_titles.csv
8360,netflix,s6911,HALO Legends,"Shinji Aramaki, Mamoru Oshii, Hideki Futamura,...",12,netflix_titles.csv
7287,netflix,s5838,X: Past Is Present,"Hemant Gaba, Pratim D. Gupta, Sudhish Kamath, ...",11,netflix_titles.csv
4574,netflix,s3125,"Sincerely Yours, Dhaka","Nuhash Humayun, Syed Ahmed Shawki, Rahat Rahma...",11,netflix_titles.csv
8614,netflix,s7165,Kahlil Gibran's The Prophet,"Roger Allers, Gaëtan Brizzi, Paul Brizzi, Joan...",10,netflix_titles.csv
4935,netflix,s3486,Sturgill Simpson Presents Sound & Fury,"Jumpei Mizusaki, Koji Morimoto, Michael Arias,...",10,netflix_titles.csv
8074,netflix,s6625,Don Quixote: The Ingenious Gentleman of La Mancha,"Mahin Ibrahim, Austin Kolodney, Will Lowell, D...",10,netflix_titles.csv
1747,netflix,s298,Navarasa,"Bejoy Nambiar, Priyadarshan, Karthik Narain, V...",9,netflix_titles.csv
8434,netflix,s6985,Holidays,"Anthony Scott Burns, Nicholas McCarthy, Adam E...",9,netflix_titles.csv



Top listed_in by count


,platform,show_id,title,listed_in,listed_in_count,source_file
10258,netflix,s8807,Zubaan,"Dramas, International Movies, Music & Musicals",3,netflix_titles.csv
10234,netflix,s8783,Yoga Hosers,"Comedies, Horror Movies, Independent Movies",3,netflix_titles.csv
10231,netflix,s8780,Yes or No 2.5,"International Movies, LGBTQ Movies, Romantic M...",3,netflix_titles.csv
10230,netflix,s8779,Yes or No 2,"International Movies, LGBTQ Movies, Romantic M...",3,netflix_titles.csv
10229,netflix,s8778,Yes or No,"International Movies, LGBTQ Movies, Romantic M...",3,netflix_titles.csv
10226,netflix,s8775,يوم الدين,"Dramas, Independent Movies, International Movies",3,netflix_titles.csv
10225,netflix,s8774,Yanda Kartavya Aahe,"Comedies, Dramas, International Movies",3,netflix_titles.csv
31,disney_plus,s32,Shang-Chi and The Legend of The Ten Rings,"Action-Adventure, Fantasy, Superhero",3,disney_plus_titles.csv
29,disney_plus,s30,Paperman,"Animation, Comedy, Family",3,disney_plus_titles.csv
24,disney_plus,s25,Jungle Cruise,"Action-Adventure, Comedy, Family",3,disney_plus_titles.csv



Top country by count


,platform,show_id,title,country,country_count,source_file
1096,disney_plus,s1097,Mulan II,"United States, South Korea, Singapore, Russia,...",15,disney_plus_titles.csv
7683,netflix,s6234,Barbecue,"Australia, Armenia, Japan, Jordan, Mexico, Mon...",12,netflix_titles.csv
757,disney_plus,s758,Bonkers,"United States, Hong Kong, South Korea, France,...",11,disney_plus_titles.csv
9854,netflix,s8404,The Look of Silence,"Denmark, Indonesia, Finland, Norway, United Ki...",10,netflix_titles.csv
2999,netflix,s1550,The Professor and the Madman,"Ireland, France, Iceland, United States, Mexic...",8,netflix_titles.csv
2956,netflix,s1507,Shaun the Sheep,"United Kingdom, Finland, Germany, United State...",8,netflix_titles.csv
57,disney_plus,s58,Thumbelina,"Ireland, United States, Canada, United Kingdom...",8,disney_plus_titles.csv
9705,netflix,s8255,The Congress,"Israel, Germany, Poland, Luxembourg, Belgium, ...",7,netflix_titles.csv
4012,netflix,s2563,Arctic Dogs,"India, United Kingdom, China, Canada, Japan, S...",7,netflix_titles.csv
9676,netflix,s8226,The Breadwinner,"Ireland, Canada, Luxembourg, United States, Un...",7,netflix_titles.csv


## Espacios / tokens vacios

In [18]:
def tokenize(x):
    if pd.isna(x) or str(x).strip()=="":
        return []
    return [p.strip() for p in str(x).split(",")]

for c in multi_cols:
    tokens = df[c].apply(tokenize)
    empty_tokens = tokens.apply(lambda lst: sum(1 for t in lst if t == ""))
    print(c, "- rows with empty tokens:", int((empty_tokens > 0).sum()))

cast - rows with empty tokens: 0
director - rows with empty tokens: 0
listed_in - rows with empty tokens: 0
country - rows with empty tokens: 7


## Valores más frecuentes

In [19]:
from collections import Counter

def top_tokens(series, n=15):
    counter = Counter()
    for x in series.dropna():
        for t in tokenize(x):
            if t:
                counter[t] += 1
    return counter.most_common(n)

for c in multi_cols:
    print(f"\nTop tokens for {c}:")
    for item, cnt in top_tokens(df[c], n=15):
        print(f"{item}: {cnt}")


Top tokens for cast:
Anupam Kher: 44
Jim Cummings: 42
Shah Rukh Khan: 35
Julie Tejwani: 33
Naseeruddin Shah: 32
Takahiro Sakurai: 32
Rupa Bhimani: 31
Akshay Kumar: 30
Om Puri: 30
Yuki Kaji: 29
Tony Hale: 28
Fred Tatasciore: 28
Amitabh Bachchan: 28
Paresh Rawal: 28
Vincent Tong: 27

Top tokens for director:
Rajiv Chilaka: 22
Jan Suter: 21
Raúl Campos: 19
Jack Hannah: 17
Paul Hoen: 16
Wilfred Jackson: 16
John Lasseter: 16
Jay Karas: 16
Suhas Kadav: 16
Marcus Raboy: 16
Robert Vince: 14
Robert Stevenson: 13
Clyde Geronimi: 13
Cathy Garcia-Molina: 13
Charles Nichols: 12

Top tokens for listed_in:
International Movies: 2752
Dramas: 2427
Comedies: 1674
International TV Shows: 1351
Documentaries: 868
Action & Adventure: 859
TV Dramas: 763
Independent Movies: 756
Children & Family Movies: 641
Family: 632
Romantic Movies: 616
TV Comedies: 581
Thrillers: 577
Animation: 542
Comedy: 526

Top tokens for country:
United States: 4873
India: 1051
United Kingdom: 907
Canada: 522
France: 415
Japan: 328


## Columnas multi-valued (`cast`, `director`, `listed_in`, `country`)

**Hallazgos esperables:**
- Estas columnas contienen listas separadas por coma, con variabilidad en cantidad de elementos por fila.
- Se detectan potenciales problemas de calidad típicos: espacios extra, tokens vacíos y valores faltantes.

**Implicancias:**
- Consultas como “top actores” o “top géneros” son más confiables si se normaliza a un modelo relacional (tablas puente).
- Operar sobre strings crudos puede inflar/ensuciar conteos si no se aplica `TRIM()`/normalización.

**Solución propuesta:**
- Normalizar a dimensiones y tablas puente:
  - `person` + `title_person` (cast/director)
  - `genre` + `title_genre` (listed_in)
  - `country_dim` + `title_country` (country)
- Aplicar limpieza mínima: `strip`, eliminar tokens vacíos, opcionalmente estandarizar casing.

## Acciones recomendadas 

A partir de los controles QA, se recomienda:

1. **Modelo normalizado** para columnas multi-valued:
   - `person` + `title_person` (actor/director)
   - `genre` + `title_genre`
   - `country_dim` + `title_country`

2. **Campos derivados** para evitar ambigüedades:
   - `duration_raw` + (`duration_minutes` / `seasons`)
   - `date_added_raw` + `date_added_parsed`

3. **Reglas de calidad básicas** (tests):
   - `platform + show_id` único
   - `type` en {Movie, TV Show}
   - Movie → `duration_minutes` no nulo; TV Show → `seasons` no nulo
   - `release_year` en rango razonable